In [ ]:
import pandas as pd

## load raw data and get a first overview

In [ ]:
df = pd.read_csv("data-scrap\\data\\query_ساینا_details.csv")

df

In [ ]:
df.info()

In [ ]:
df.describe()

## filter out old ads

In [ ]:
import re

def persian_to_english_digits(text):
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return text.translate(trans_table)

def parse_posted_days_ago(text):
    """
    converts posted_time_text into an approximate number of days ago.
    
    handles the patterns actually observed in real Divar data:
      - دقایقی پیش" / "لحظاتی پیش"  ->       0"
      - دیروز"                       ->       1"
      - پریروز"                      ->       2
      
      - "<N> ساعت پیش"                                   -> zero
      - "<N> روز پیش"                                    -> N
      - "<N> هفته پیش" (or bare "هفته پیش" = 1 هفته)    -> N*7
      - "<N> ماه پیش"  (or bare)                         -> N*30
      - "<N> سال پیش"  (or bare)                         -> N*365
 
    the location part after the time phrase (e.g. "در تهران، نارمک جنوبی")
    is ignored. only the leading time phrase is parsed.
    """
    if pd.isna(text):
        return None

    text = persian_to_english_digits(str(text).strip())

    if text.startswith("دقایقی") or text.startswith("لحظاتی"):
        return 0
    if text.startswith("دیروز"):
        return 1
    if text.startswith("پریروز"):
        return 2

    match = re.match(r"^(\d+)?\s*(ساعت|روز|هفته|ماه|سال)\s*پیش", text)
    if not match:
        return None

    number = int(match.group(1)) if match.group(1) else 1
    unit = match.group(2)

    unit_to_days = {
        "ساعت": 0,
        "روز": 1,
        "هفته": 7,
        "ماه": 30,
        "سال": 365,
    }
    return number * unit_to_days[unit]

In [ ]:
df["days_ago"] = df["posted_time_text"].apply(parse_posted_days_ago)
df[["posted_time_text", "days_ago"]]

In [ ]:
df["scraped_at"] = pd.to_datetime(df["scraped_at"])
df["actual_post_date"] = df["scraped_at"] - pd.to_timedelta(df["days_ago"], unit="D")

df[["posted_time_text", "days_ago", "scraped_at", "actual_post_date"]]

In [ ]:
now = pd.Timestamp.now(tz=df["actual_post_date"].dt.tz)
df["ad_age_days_from_now"] = (now - df["actual_post_date"]).dt.total_seconds() / 86400

df[["actual_post_date", "ad_age_days_from_now"]]

In [ ]:
MAX_AGE_DAYS = 30

df["is_recent"] = df["ad_age_days_from_now"]<=MAX_AGE_DAYS

print(f"Total: {len(df)}")
print(f"Recent (<= {MAX_AGE_DAYS} days): {df['is_recent'].sum()}")
print(f"Too old: {(~df['is_recent']).sum()}")

df = df[df["is_recent"]].copy()
df.reset_index(drop=True, inplace=True)

df

## parse `year_text` into numeric Jalali/Gregorian years

In [ ]:
year_col_idx = df.columns.get_loc("year_text")

df.insert(year_col_idx+1, "model_year", df["year_text"].apply(persian_to_english_digits))

df.insert(
    year_col_idx+2,
    "miladi_year",
    pd.to_numeric(df["model_year"].str.extract(r"-\s*(\d{4})")[0], errors="coerce")
)

df.insert(
    year_col_idx+3,
    "jalali_year",
    pd.to_numeric(df["model_year"].str.extract(r"(\d{4})\s*-")[0], errors="coerce")
)

df

## remove "havale" (pre-order) listings

In [ ]:
import datetime
current_year = datetime.date.today().year

havale_idx = df[
                (df["title"].str.contains("حواله|اماده|آماده|تحویل", na=False))
                & (df["mileage_km"] == 0)
                & ( (df["miladi_year"] == current_year) | (df["miladi_year"] == current_year-1) | (df["miladi_year"] == current_year-2) )
            ].index

# df.loc[havale_idx]

# df = df.drop(havale_idx)
df.drop(havale_idx, inplace=True)

df.reset_index(drop=True, inplace=True)

df

In [ ]:
df.info()

## fill missing `gearbox` from the model name

In [ ]:
def get_gearbox(row):
    if pd.notna(row["gearbox"]):
        return row["gearbox"]

    model = row["brand_model_text"]

    if pd.isna(model):
        return None

    model = str(model).replace("ي", "ی").replace("ك", "ک")
    model = model.replace("‌", " ")

    if "اتومات" in model:
        return "اتومات"

    if "دنده" in model or "دستی" in model:
        return "دنده‌ای"

    return None

df["gearbox"] = df.apply(get_gearbox, axis=1)

df.info()

In [ ]:
df[df["gearbox"].isna()]

In [ ]:
df.drop(df[df["gearbox"].isna()].index, inplace=True)

df.reset_index(drop=True, inplace=True)

df.info()
# df

## extract city from `posted_time_text`

In [ ]:
CITY_NAME = ["تهران",]  # "شیراز", "اصفهان", "بوشهر"

col_idx = df.columns.get_loc("location")

df.insert(
    col_idx + 1,
    "city_location_persian",
    df["posted_time_text"].apply(lambda x: next((city for city in CITY_NAME if city in x), None))  # CITY_NAME if CITY_NAME in x else None
)

df

## build the working subset (`fdf`)

In [ ]:
fdf = df[["token", "brand_model_text", "color", "jalali_year", "miladi_year", "gearbox", "fuel_type", "mileage_km", "engine_condition", "chassis_front_condition", "chassis_rear_condition", "body_condition", "gearbox_condition", "base_price_toman"]].copy()  # "city_location_persian"
fdf

In [ ]:
fdf.columns, fdf.info()

## handle the condition columns

In [ ]:
import numpy as np

condition_cols = [
    "engine_condition",
    "chassis_front_condition",
    "chassis_rear_condition",
    "body_condition",
    "gearbox_condition",
]

fdf[condition_cols] = fdf[condition_cols].replace("تعیین‌نشده", np.nan)

fdf.info()

#### fill missing conditions for zero-mileage cars using the most common "سالم" value

In [ ]:
for col in condition_cols:
    values = fdf[col].value_counts()
    
    healthy_values = values[values.index.str.contains("سالم", na=False)]
    healthy_value = healthy_values.index[0] if len(healthy_values)>0 else np.nan

    fdf.loc[(fdf["mileage_km"]==0) & (fdf[col].isna()), col] = healthy_value

fdf.info()

#### fill remained missing conditions using the description

In [ ]:
# missing_matrix = fdf[condition_cols].isna()
# missing_matrix

In [ ]:
# fdf["missing_count"] = missing_matrix.sum(axis=1)
# fdf[fdf["missing_count"] > 0].sort_values("missing_count", ascending=False)[
#     ["token", "brand_model_text", "mileage_km"] + condition_cols + ["missing_count"]
# ]

In [ ]:
# missing_tokens = fdf.loc[fdf["missing_count"] > 0, "token"]

# for token in missing_tokens[:33]:
#     desc = df.loc[df["token"] == token, "description"].values[0]
#     print(f"=== {token} ===")
#     print(desc)
#     print()

no clear, predictable pattern here. trying to keyword-match the wording would likely add noise rather than reduce error.

#### so filling the remaining missing values with "unknown". the model can learn it as its own category, and this also prepares it to handle unknown values in future data

In [ ]:
fdf[condition_cols] = fdf[condition_cols].fillna("نامشخص")

fdf.info()

## exploratory data analysis

In [ ]:
fdf["base_price_toman"].hist()

In [ ]:
features = [
    "brand_model_text",
    "jalali_year",
    "miladi_year",
    "gearbox",
    "fuel_type",
    "mileage_km",
    "engine_condition",
    "chassis_front_condition",
    "chassis_rear_condition",
    "body_condition",
    "gearbox_condition",
    # "base_price_toman"
]


for feature in features:
    print(f"\n{fdf[feature].value_counts()}")

In [ ]:
import matplotlib.pyplot as plt
from math import ceil

target = "base_price_toman"

cols = 3
rows = ceil(len(features) / cols)
# fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4), gridspec_kw={"hspace": 0.6})

# for ax, feature in zip(axes.flat, features):
#     ax.scatter(fdf[feature], fdf[target], color="blue")
#     ax.set_xlabel(feature)
#     ax.set_ylabel(target)
#     ax.set_title(f"{feature} vs {target}")

# plt.tight_layout()
# plt.show()

In [ ]:
numeric_features = ["jalali_year", "miladi_year", "mileage_km"]
categorical_features = [
    "brand_model_text", "gearbox", "fuel_type", "engine_condition",
    "chassis_front_condition", "chassis_rear_condition",
    "body_condition", "gearbox_condition",
]

# scatter برای numeric
fig, axes = plt.subplots(1, len(numeric_features), figsize=(15, 4))
for ax, feature in zip(axes, numeric_features):
    ax.scatter(fdf[feature], fdf[target], alpha=0.5)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)

plt.tight_layout()
plt.show()

# boxplot برای categorical
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feature in zip(axes.flat, categorical_features):
    fdf.boxplot(column=target, by=feature, ax=ax, rot=45)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

### check for unrealistic high prices

In [ ]:
fdf.sort_values("base_price_toman", ascending=False)[
    ["token", "brand_model_text", "mileage_km", "jalali_year", "base_price_toman"]
].head(15)

### drop the extreme outliers

In [ ]:
err  # comment this line, check extreme outlier index, and replace CHANGE_IT with the index of the extreme outlier to drop it from the DataFrame
extreme_outlier_index=CHANGE_IT
fdf.drop(index=extreme_outlier_index, inplace=True)

### re-run after removing the extreme outliers

In [ ]:
fdf.sort_values("base_price_toman", ascending=False)[
    ["token", "brand_model_text", "mileage_km", "jalali_year", "base_price_toman"]
].head(15)

In [ ]:
numeric_features = ["jalali_year", "miladi_year", "mileage_km"]
categorical_features = [
    "brand_model_text", "gearbox", "fuel_type", "engine_condition",
    "chassis_front_condition", "chassis_rear_condition",
    "body_condition", "gearbox_condition",
]

# scatter برای numeric
fig, axes = plt.subplots(1, len(numeric_features), figsize=(15, 4))
for ax, feature in zip(axes, numeric_features):
    ax.scatter(fdf[feature], fdf[target], alpha=0.5)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)

plt.tight_layout()
plt.show()

# boxplot برای categorical
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feature in zip(axes.flat, categorical_features):
    fdf.boxplot(column=target, by=feature, ax=ax, rot=45)
    ax.set_title(feature)

plt.tight_layout()
plt.show()

In [ ]:
fdf["base_price_toman"].hist()

## check numeric correlation with price

In [ ]:
numeric_features = ["jalali_year", "mileage_km"]
print("Correlation with price:")
print(fdf[numeric_features + ["base_price_toman"]].corr()["base_price_toman"])

In [ ]:
categorical_features = [
    "gearbox", "fuel_type", "engine_condition",
    "chassis_front_condition", "chassis_rear_condition",
    "body_condition", "gearbox_condition",
]

for feature in categorical_features:
    group_means = fdf.groupby(feature)["base_price_toman"].mean().sort_values()
    print(f"\n{feature}:")
    print(group_means)

In [ ]:
fdf

# try to train..

### outlier detection with statics

##### IQR

In [ ]:
def flag_iqr_outliers(df, target, multiplier=1.5):
    q1 = df[target].quantile(0.25)
    q3 = df[target].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier*iqr
    upper = q3 + multiplier*iqr
    return (df[target] < lower) | (df[target] > upper)

# def flag_iqr_outliers_by_group(df, group_cols, target_col, multiplier=1.5):
#     def flag_group(group):
#         q1 = group[target_col].quantile(0.25)
#         q3 = group[target_col].quantile(0.75)
#         iqr = q3 - q1
#         lower = q1 - multiplier*iqr
#         upper = q3 + multiplier*iqr
#         return (group[target_col] < lower) | (group[target_col] > upper)
#     return df.groupby(group_cols, group_keys=False).apply(flag_group)

# def flag_iqr_outliers_on_residual(df, residual_col, multiplier=1.5):
#     q1 = df[residual_col].quantile(0.25)
#     q3 = df[residual_col].quantile(0.75)
#     iqr = q3 - q1
#     lower = q1 - multiplier*iqr
#     upper = q3 + multiplier*iqr
#     return (df[residual_col] < lower) | (df[residual_col] > upper)

In [ ]:
fdf["is_iqr_outlier"] = flag_iqr_outliers(fdf, target, 1.5)

fdf[fdf["is_iqr_outlier"]]

##### Z score

In [ ]:
from scipy import stats

fdf["zscore"] = stats.zscore(fdf[target])
fdf["is_zscore_outlier"] = fdf["zscore"].abs() > 3

fdf[fdf["is_zscore_outlier"]]

##### modified Z score

In [ ]:
median = fdf[target].median()
mad = (fdf[target] - median).abs().median()
fdf["modified_zscore"] = 0.6745 * (fdf[target] - median) / mad
fdf["is_modified_zscore_outlier"] = fdf["modified_zscore"].abs() > 3.5

fdf[fdf["is_modified_zscore_outlier"]]

### outlier detection with ML models

##### data preprocessing

In [ ]:
from sklearn.preprocessing import OneHotEncoder, RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score

In [ ]:
numeric_features = ["jalali_year", "mileage_km"]
categorical_features = ["gearbox", "fuel_type", "engine_condition", "chassis_front_condition", "chassis_rear_condition", "body_condition", "gearbox_condition"]
target = "base_price_toman"

In [ ]:
for f in categorical_features:
    print(f, fdf[f].unique(), '\n')

In [ ]:
ordinal_categories = {
    "engine_condition": ["نیاز به تعمیر", "نامشخص", "سالم"],
    "chassis_front_condition": ["ضربه‌خورده", "نامشخص", "سالم و پلمپ"],
    "chassis_rear_condition": ["ضربه‌خورده", "نامشخص", "سالم و پلمپ"],
    "gearbox_condition": ["نیاز به تعمیر جزئی", "تعمیر شده", "نامشخص", "سالم و پلمپ"],
    "body_condition": [ "تصادفی", "دوررنگ", "رنگ‌شدگی در ۳ ناحیه", "رنگ‌شدگی در ۲ ناحیه", "رنگ‌شدگی در ۱ ناحیه", "صافکاری بی‌رنگ", "خط و خش جزیی", "سالم و بی‌خط و خش",],
}

ordinal_features = list(ordinal_categories.keys())
nominal_features = ["gearbox", "fuel_type"]

In [ ]:
outlier_preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", OrdinalEncoder(categories=[ordinal_categories[col] for col in ordinal_features]), ordinal_features),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("num", RobustScaler(), numeric_features + [target]),
    ]
)

X_outlier = fdf[numeric_features + categorical_features + [target]]
X_outlier_encoded = outlier_preprocessor.fit_transform(X_outlier)
X = X_outlier_encoded.toarray() if hasattr(X_outlier_encoded, "toarray") else X_outlier_encoded

X.shape

In [ ]:
ordinal_encoder = outlier_preprocessor.named_transformers_["ordinal"]

for col, categories in zip(ordinal_features, ordinal_encoder.categories_):
    print(f"{col}:")
    for i, cat in enumerate(categories):
        print(f"  {i} -> {cat}")
    print()

In [ ]:
X